In [1]:
import os

GIT = True
REPO_NAME = "generative_models_for_image"

if GIT:
    if not os.path.exists(REPO_NAME):
        !git clone https://github.com/Mayeul84/generative_models_for_image

import os
os.chdir(REPO_NAME)

Cloning into 'generative_models_for_image'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 102 (delta 10), reused 94 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 7.57 MiB | 7.07 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [ ]:
import torch
import torchvision
import numpy as np

import matplotlib.pyplot as plt

import yaml
from PIL import Image
from tqdm import tqdm
from guided_diffusion.unet import create_model

from utils_image import *

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

!wget -nc -O data/ffhq256-1k-validation.zip 'https://www.dropbox.com/scl/fi/pppstbdsf0em6o0qscruc/ffhq256-1k-validation.zip?rlkey=xl7nwv2nxb6yvsirr3wad77hm'
!unzip -nq data/ffhq256-1k-validation.zip

!wget -nc -O models/ffhq_10m.pt 'https://www.dropbox.com/scl/fi/pq72vxzxcbygieq5z4gvf/ffhq_10m.pt?rlkey=5sxdj6r4o9f7b7bbp5fxg2f5r'

In [ ]:
def load_yaml(file_path: str) -> dict:
    with open(file_path) as f:
        config = yaml.load(f, Loader=yaml.FullLoader)
    return config

args = {
    'data_config': 'configs/data_config.yaml',
    'model_config': 'configs/model_config.yaml',
    'diffusion_config': 'configs/diffusion_config.yaml',
    'operator_config': 'configs/operator_config.yaml',
    'gibbs_config': 'configs/gibbs_config.yaml',
    'gpu': 0,
    'save_dir': './results'
}

data_config = load_yaml(args['data_config'])
model_config = load_yaml(args['model_config'])
diffusion_config = load_yaml(args['diffusion_config'])
operator_config = load_yaml(args['operator_config'])
gibbs_config = load_yaml(args['gibbs_config'])

In [ ]:
# # Load model
# model = create_model(**model_config)
# model = model.to(device)
# model.eval()

# Load model
model_config = {'image_size': 256,
                'num_channels': 128,
                'num_res_blocks': 1,
                'channel_mult': '',
                'learn_sigma': True,
                'class_cond': False,
                'use_checkpoint': False,
                'attention_resolutions': 16,
                'num_heads': 4,
                'num_head_channels': 64,
                'num_heads_upsample': -1,
                'use_scale_shift_norm': True,
                'dropout': 0.0,
                'resblock_updown': True,
                'use_fp16': False,
                'use_new_attention_order': False,
                'model_path': 'models/ffhq_10m.pt'}
model = create_model(**model_config)
model = model.to(device)
model.eval();  # use in eval mode

In [ ]:
### Choose the image
idx = 462
img_pil = Image.open('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png')
x0 = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'),device=device)
imgshape = x0.shape

In [ ]:
idx = np.random.randint(1000)
print('Image', str(idx).zfill(5))

# Test display function:
img = im2tensor(plt.imread('ffhq256-1k-validation/'+str(idx).zfill(5)+'.png'),device=device)
print(img.min().item(),img.max().item())
viewimage(img)

In [ ]:
T = 1000
time_steps = np.arange(T)
reversed_time_steps = time_steps[::-1]

beta_start = 0.0001
beta_end = 0.02
beta = np.linspace(beta_start, beta_end, T, dtype=np.float64)
alpha = 1.0 - beta
alphabar = np.cumprod(alpha)
betabar = 1-alphabar
alphabar_prev = np.append(1.0, alphabar[:-1])    # alpha_{t-1}

def get_eps_from_model(x, t):
    # the model outputs:
    # - an estimation of the noise eps (chanels 0 to 2)
    # - learnt variances for the posterior  (chanels 3 to 5)
    # (see Improved Denoising Diffusion Probabilistic Models
    # by Alex Nichol, Prafulla Dhariwal
    # for the parameterization)
    # We discard the second part of the output for this practice session.

    with torch.no_grad(): # avoid backprop wrt model parameters
        model_output = model(x, torch.tensor(t, device=device).unsqueeze(0))
    model_output = model_output[:,:3,:,:]
    return(model_output)


def get_eps_from_model_withgrad(x, t):
    # same as get_eps_from_model but kipping gradients
    model_output = model(x, torch.tensor(t, device=device).unsqueeze(0))
    model_output = model_output[:,:3,:,:]
    return(model_output)

In [ ]:
h = imgshape[2]
w = imgshape[3]
hcrop, wcrop = h//2, w//2
corner_top, corner_left = h//4, int(0.45*w)
mask = torch.ones(imgshape, device=device)
mask[:,:,corner_top:corner_top+hcrop,corner_left:corner_left+wcrop] = 0

def linear_operator(x):
  x = x*mask
  return(x)

x_true = x0.clone()
print("original image")
viewimage(x_true)

sigma_noise = 2*10/255

y = linear_operator(x_true.clone()) + sigma_noise * mask * torch.randn_like(x_true)
print("noisy measurement")
viewimage(y)

In [ ]:
# visualization image for the observation y  (for inpainting, no resizing required)
vis_y = y

# initialize xt for t=T
x = torch.randn(imgshape, device=device,requires_grad=True)


for i, t in enumerate(reversed_time_steps):

  # TODO
  eps_theta = get_eps_from_model_withgrad(x,t)
  mu = 1/np.sqrt(alpha[t]) * (x - beta[t]/np.sqrt(betabar[t]) * eps_theta)
  xhat = 1/np.sqrt(alphabar[t])*x - np.sqrt(1/alphabar[t] - 1) * eps_theta
  zeta = 0.1/torch.norm(linear_operator(xhat) - y)
  grad_x = torch.autograd.grad(
        outputs=torch.norm(linear_operator(xhat) - y)**2,
        inputs=x
  )[0]


  x = mu + np.sqrt(beta[t])*torch.randn(size=x.shape,device=device) - zeta*grad_x

  # Display
  if i==0 or t%100==0 or t==0:
      print('Iteration:', i, '; Discrete time:', t)
      viewimage(torch.cat((x, xhat, vis_y), dim=3))